# 08 · Neural Network Mathematics

**The big idea:** a neural network is `y = f(Wx + b)` repeated. Training = forward pass, measure loss, backprop (chain rule), update weights.

**Where this leads →** a Transformer is the same pieces with attention added (09).

---

## 📂 Notes in this chapter

One short note per concept, grouped into folders. Read in order. Code builds on earlier notes, so `notebook.ipynb` (all code, in order) is the easiest way to run everything.

### `1-neuron-layers-forward/`

- [1. The Neuron & Weighted Sum](1-neuron-layers-forward/01-the-neuron-weighted-sum.md)
- [2. Activation Functions](1-neuron-layers-forward/02-activation-functions.md)
- [3. Forward Propagation](1-neuron-layers-forward/03-forward-propagation.md)

### `2-loss-and-backprop/`

- [4. Loss](2-loss-and-backprop/04-loss.md)
- [5. Backpropagation & the Chain Rule](2-loss-and-backprop/05-backpropagation-the-chain-rule.md)

### `3-updates-and-training-loop/`

- [6. Weight Updates](3-updates-and-training-loop/06-weight-updates.md)
- [7. The Training Loop](3-updates-and-training-loop/07-the-training-loop.md)


# The Neuron & Weighted Sum

*📓 Notes · [08 · Neural Network Mathematics](../README.md) · neuron layers forward*

**📝 In simple words:** Take inputs, weight them, add a bias.

**💡 Think of it like this:** Each weight says "how much do I care about this input?"

**🔢 Small example:** `x = [1, 2]`, `w = [0.5, −1]`, `b = 0.5`.

**📐 Formula to remember:** `z = w·x + b`

**✏️ Let me calculate it:** `0.5·1 + (−1)·2 + 0.5 = −1`.

```text
 x₁ ──w₁──╲
            ( Σ + b ) ──► activation ──► output
 x₂ ──w₂──╱
```

**🤖 Why AI uses it:** it's the dot product (01) plus a bias. A layer = many neurons = `Wx + b`.

In [ ]:
import numpy as np
x = np.array([1., 2.]); w = np.array([0.5, -1.]); b = 0.5
z = w @ x + b; print(z)   # -1.0

**🧪 Try it myself:** `w=[2,1]`, `x=[3,−4]`, `b=1`: z? <details><summary>Solution</summary>`6 − 4 + 1 = 3`.</details>

---
[Activation Functions](../1-neuron-layers-forward/02-activation-functions.md) →


# Activation Functions

*📓 Notes · [08 · Neural Network Mathematics](../README.md) · neuron layers forward*

**📝 In simple words:** A non-linear squash applied to `z`.

**💡 Think of it like this:** Without it, stacked layers collapse into one linear map (matrix × matrix = matrix). The bend lets networks learn curves.

| Name | Formula | Range | Used for |
|---|---|---|---|
| Sigmoid | `1/(1+e⁻ᶻ)` | (0,1) | binary output, gates |
| Tanh | `(eᶻ−e⁻ᶻ)/(eᶻ+e⁻ᶻ)` | (−1,1) | older RNNs |
| ReLU | `max(0, z)` | [0,∞) | hidden layers |
| GELU | `z·Φ(z)` (smooth ReLU) | ≈[−0.17,∞) | Transformers |

**✏️ Let me calculate it (`z = −1`):** sigmoid `0.269`, tanh `−0.762`, ReLU `0`.

In [ ]:
z = -1.0
print(1/(1+np.exp(-z)), np.tanh(z), max(0, z))

**🧪 Try it myself:** ReLU(3) and ReLU(−3)? <details><summary>Solution</summary>3 and 0.</details>

---
← [The Neuron & Weighted Sum](../1-neuron-layers-forward/01-the-neuron-weighted-sum.md)  |  [Forward Propagation](../1-neuron-layers-forward/03-forward-propagation.md) →


# Forward Propagation

*📓 Notes · [08 · Neural Network Mathematics](../README.md) · neuron layers forward*

**📝 In simple words:** Pushing the input through the layers to get a prediction.

**🔢 Small example:** one input `x=1`; hidden: `h = ReLU(2x) `; output `ŷ = 0.5h`.

**📐 Formula to remember:** `h = ReLU(W₁x + b₁)`, `ŷ = W₂h + b₂`

**✏️ Let me calculate it:** `h = ReLU(2) = 2`; `ŷ = 0.5·2 = 1`.

**🤖 Why AI uses it:** this is inference: what a trained model does every time you use it.

In [ ]:
x, w1, w2 = 1.0, 2.0, 0.5
h = max(0, w1 * x); y_hat = w2 * h
print(h, y_hat)   # 2.0 1.0

**🧪 Try it myself:** Same net with `x = −1`? <details><summary>Solution</summary>`h = ReLU(−2) = 0`, so `ŷ = 0`.</details>

---
← [Activation Functions](../1-neuron-layers-forward/02-activation-functions.md)  |  [Loss](../2-loss-and-backprop/04-loss.md) →


# Loss

*📓 Notes · [08 · Neural Network Mathematics](../README.md) · loss and backprop*

**📝 In simple words:** Compare prediction to truth, one number out (chapter 05/07).

**🔢 Small example:** `y = 3`, `ŷ = 1` → `L = (ŷ − y)² = 4`.

**Softmax + cross-entropy gem:** for probabilities `p` and one-hot label `y`, the gradient of the loss with respect to the logits is just **`p − y`**. Example `p = [0.665, 0.245, 0.090]`, `y = [1,0,0]` → gradient `[−0.335, 0.245, 0.090]`: push class 1 up, others down.

In [ ]:
p = np.array([0.665, 0.245, 0.090]); y = np.array([1, 0, 0])
print(p - y)

**🧪 Try it myself:** `p=[0.2, 0.8]`, `y=[0,1]`: gradient? <details><summary>Solution</summary>`[0.2, −0.2]`.</details>

---
← [Forward Propagation](../1-neuron-layers-forward/03-forward-propagation.md)  |  [Backpropagation & the Chain Rule](../2-loss-and-backprop/05-backpropagation-the-chain-rule.md) →


# Backpropagation & the Chain Rule

*📓 Notes · [08 · Neural Network Mathematics](../README.md) · loss and backprop*

**📝 In simple words:** Computing every weight's gradient by walking backward through the network.

**💡 Think of it like this:** The loss blames the output; the output blames the layer before it, and so on. Each blame is a multiplication of local slopes: the chain rule (02).

**📐 Formula to remember:** `∂L/∂w₁ = ∂L/∂ŷ · ∂ŷ/∂h · ∂h/∂w₁`

**🔢 Small example:** the net from step 3 with `y = 3`, so `L = (ŷ − y)²`.

**✏️ Let me calculate it:**

```text
forward:  h = 2,  ŷ = 1,  L = (1−3)² = 4
∂L/∂ŷ  = 2(ŷ − y)     = −4
∂L/∂w₂ = ∂L/∂ŷ · h    = −4 · 2   = −8
∂L/∂h  = ∂L/∂ŷ · w₂   = −4 · 0.5 = −2
∂L/∂w₁ = ∂L/∂h · 1(ReLU′, since z>0) · x = −2
```

**🤖 Why AI uses it:** this is how *every* deep network learns, from LeNet to GPT.

In [ ]:
x, y, w1, w2 = 1.0, 3.0, 2.0, 0.5
h = max(0, w1*x); y_hat = w2*h
dL_dyh = 2*(y_hat - y)
dw2 = dL_dyh * h
dw1 = dL_dyh * w2 * (1.0 if w1*x > 0 else 0.0) * x
print(dw1, dw2)   # -2.0 -8.0
# gradient check: nudge and measure
def L(w1, w2): return (w2*max(0, w1*x) - y)**2
print((L(w1+1e-6, w2) - L(w1, w2)) / 1e-6)   # ≈ -2

**🧪 Try it myself:** Why is gradient 0 for a ReLU unit whose `z < 0`? <details><summary>Solution</summary>ReLU′ is 0 there, so the product is 0 and the weight gets no signal ("dead ReLU").</details>

---
← [Loss](../2-loss-and-backprop/04-loss.md)  |  [Weight Updates](../3-updates-and-training-loop/06-weight-updates.md) →


# Weight Updates

*📓 Notes · [08 · Neural Network Mathematics](../README.md) · updates and training loop*

**📐 Formula to remember:** `w ← w − α · ∂L/∂w`

**✏️ Let me calculate it (α = 0.1):** `w₂ = 0.5 − 0.1(−8) = 1.3`; `w₁ = 2 − 0.1(−2) = 2.2`. New forward: `h = 2.2`, `ŷ = 2.86`, `L = 0.0196` (down from 4).

In [ ]:
lr = 0.1
w2 -= lr * dw2; w1 -= lr * dw1
print(w1, w2, w2*max(0, w1*x))   # 2.2 1.3 2.86

**🧪 Try it myself:** If a gradient is positive, does the weight go up or down? <details><summary>Solution</summary>Down. We move against the gradient.</details>

---
← [Backpropagation & the Chain Rule](../2-loss-and-backprop/05-backpropagation-the-chain-rule.md)  |  [The Training Loop](../3-updates-and-training-loop/07-the-training-loop.md) →


# The Training Loop

*📓 Notes · [08 · Neural Network Mathematics](../README.md) · updates and training loop*

```text
repeat:
  1. forward pass     → predictions
  2. loss             → how wrong
  3. backward pass    → gradients
  4. update weights   → w ← w − α·grad
```

A complete tiny network learning XOR (a problem no single line can solve), manual gradients only:

In [ ]:
rng = np.random.default_rng(1)
X = np.array([[0,0],[0,1],[1,0],[1,1]], float); Y = np.array([[0],[1],[1],[0]], float)
W1 = rng.normal(0, 1, (2, 4)); b1 = np.zeros(4)
W2 = rng.normal(0, 1, (4, 1)); b2 = np.zeros(1)
sig = lambda z: 1/(1+np.exp(-z))
for step in range(5000):
    H = np.tanh(X @ W1 + b1)                 # forward
    P = sig(H @ W2 + b2)
    loss = -np.mean(Y*np.log(P) + (1-Y)*np.log(1-P))
    dZ2 = (P - Y) / len(X)                   # backward (BCE + sigmoid → p − y)
    dW2 = H.T @ dZ2; db2 = dZ2.sum(0)
    dH = dZ2 @ W2.T; dZ1 = dH * (1 - H**2)   # tanh′ = 1 − tanh²
    dW1 = X.T @ dZ1; db1 = dZ1.sum(0)
    for p_, g in ((W1, dW1), (b1, db1), (W2, dW2), (b2, db2)):
        p_ -= 0.5 * g                        # update
print(round(loss, 4), P.round(2).ravel())    # loss ≈ 0.001, predictions ≈ [0 1 1 0]
# try seed 0: it gets stuck at loss 0.35. Bad starting weights can trap training!

**🧪 Try it myself:** In the loop, which line is "backprop"? <details><summary>Solution</summary>From `dZ2 = ...` to `db1 = ...`. These apply the chain rule layer by layer.</details>

---
← [Weight Updates](../3-updates-and-training-loop/06-weight-updates.md)